An module which performs the resume changes based upon the requirements mentioned in the job appication, here the langgraph would be implemented, which has the following variables which has to be filled: 

1. `resume_path` : The path where the **Original resume** is present. Make sure the resume is in `.tex` format as we would be changing the `.tex` using an python package in the cli.
2. `job_req` : This is the requirements of the job.


## 1 Task:

Do the code for laying down the basic flow on how the steps should be present, like "What should happen when this happens", etc.

For this implemet the LangGraph feature which follows a static workflow.

Then internally to alter the `.tex` (i.e original resume) implement the agent in there

#### Basic flow:

* Understand the resume and job requirements (Not model oriented) 

* Make a Generator Model that makes the changes according to the job statements

* Make a discriminator that defies what are the errors made and changes that could be made on the resume.

* Let those both use the agent internally to a> Perform a little lag while submitting the application which imitates the human behaviou b> More precise


In [3]:
from typing_extensions import TypedDict, Dict
from pydantic import Field

In [4]:

class Graph(TypedDict):
    resume_path:str
    job_req:str
    session_id:str
    chat_loaders:dict


In [5]:
from langchain_groq import ChatGroq
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.agents import AgentExecutor, create_tool_calling_agent

import os
model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key="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",
    temperature=0.5
)



KeyboardInterrupt: 

In [ ]:
"""HF_TOKEN=os.getenv("HF_TOKEN")
embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_docs=TextLoader("resume.txt").load()
docs=RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=20).split_documents(text_docs)
vc=FAISS.from_documents(docs,embedding)"""

In [ ]:
def read_docs (state:Graph)->Graph:
    small_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.2,
        api_key="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"
    )
    @tool
    def understand_document(to_find:str)->str:
        """Use this function to obtain the current resume data, to help you understand whether the changes have to be made or not"""
        total_text=[i.page_content for i in vc.similarity_search(to_find)]
        return total_text

    @tool
    def change_project(project_changes)->None:
        """Use this module to change the project sub section like the works i have done"""
        latex_code =r"""\resumeSubHeadingListStart
        \resumeProjectHeading
            {\textbf{PROJECT NAME} $|$ }{}
            \resumeItemListStart
                \resumeItem{text explaining the project}
            \resumeItemListEnd 
        \resumeSubHeadingListEnd"""
        prompt=ChatPromptTemplate.from_messages([("assistant", "You are a intelligent resume maker, you would be provided with the content, of what should be placed in the latex template, you have to intelligently only provide the updated latex template"),
                                                ("user", """Here is the changes that have to be made:  {project_changes}, Here is the latex code  {latex_code}
                                                Strictly generate the output like the latex code, no extra explanation needed as we would be pasting whatever you give directly in the resume latex."""),
                                                ("ai", """%-----------PROJECTS-----------
    \section""")])
            
        chain=prompt | small_llm | StrOutputParser()
        output=chain.invoke({
            "project_changes":project_changes,
            "latex_code":latex_code
        })
        with open ("updates.txt","w") as f:
            f.write(output)
        return None
    tools=[understand_document, change_project]

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
    You are an intelligent ATS and resume updation machine.
    Based upon the requirements given, implement the changes in the original resume.
    """
        ),

        ("human", "{JD}"),

        MessagesPlaceholder(variable_name="agent_scratchpad")
    ])

    agent = create_tool_calling_agent(
        llm=model,
        tools=tools,
        prompt=prompt
    )
    final_agent = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True
    )
    final_agent.invoke({

        "JD": """
    Need Engineer with Web development skills
    """
    }, config={"configurable":{"session_id":state["session_id"]}})
    return None



        
    

In [ ]:
def initial_state(state: Graph) -> Graph:

    if "chat_loaders" not in state:
        state["chat_loaders"] = {}

    def get_session_history(session_id: str) -> BaseChatMessageHistory:

        if session_id not in state["chat_loaders"]:
            state["chat_loaders"][session_id] = ChatMessageHistory()

        return state["chat_loaders"][session_id]

    dummy_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
You are a professional resume editor AI.

Your responsibility is to:

1. Analyze the given job requirements carefully.
2. Analyze the employee resume carefully.
3. Identify missing skills, weak points, ATS issues, irrelevant content, and optimization opportunities.
4. Suggest modifications that improve the resume according to the job role.

STRICTLY FOLLOW THIS OUTPUT FORMAT:

<Changes>
Mention the exact change that should be made
</Changes>

<Reason>
Explain why this change is necessary
</Reason>
"""
            ),

            MessagesPlaceholder(variable_name="history"),

            (
                "human",
                """
Here are the job requirements:

{job_req}

###################################

Here is the employee resume:

{resume_path}

###################################

{input}
"""
            )
        ]
    )

    final_chain = dummy_prompt | model

    runnable_chain = RunnableWithMessageHistory(
        final_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )

    response = runnable_chain.invoke(
        {
            "job_req": state["job_req"],
            "resume_path": state["resume_path"],
            "input": "Strictly follow the schema."
        },
        config={
            "configurable": {
                "session_id": state["session_id"]
            }
        }
    )

    state["response"] = response.content

    return state

In [ ]:
def state2(state:Graph)->Graph:
    #Understands the changes that have to be made and makes the repective changes.

    parser_prompt=ChatPromptTemplate.from_messages([("assistant", """
You are an intelligent resume modification extraction system.

Understand the given chat history carefully.

The history contains resume modification suggestions generated by another AI.

FORMAT:

Region of change -> Updated format.

RULES:
- No numbering
- **No extra explanation**
- No markdown
- No XML tags
- No keywords in headings, it has to be solid heading

CHAT HISTORY:
{history}
"""), ("ai", "Ok, here is a strict fromat you expected with the updates which would be directly changes in resume: <Changes> Title Change ->")])
    chain_dummy=parser_prompt | model | StrOutputParser()
    with open("updates.txt", "w") as f:
        f.write(chain_dummy.invoke({
            "history":state["chat_loaders"][state["session_id"]]
        }))



    prompt=ChatPromptTemplate.from_messages([("assistant", "You are a resume updater, who updates the resume based upon the changes that have been mentioned, YOU STRICTLY RETURN THE UPDATED INPUT NOTHING ELSE!!!")])

    

In [ ]:
from langgraph.graph import StateGraph,END
graph=StateGraph(Graph)
graph.add_node("init_node", initial_state)
graph.add_node("sec_node", state2)
graph.set_entry_point("init_node")
graph.add_edge("init_node", "sec_node")
graph.add_edge("sec_node", END)

In [ ]:
from langgraph.graph import StateGraph,END
graph=StateGraph(Graph)
graph.add_node("init_node", initial_state)
graph.add_node("sec_node", state2)
graph.add_node("prim1",read_docs)
graph.set_entry_point("prim1")
#graph.add_edge("init_node", "sec_node")
#graph.add_edge("sec_node", END)
graph.add_edge("prim1",END)

In [ ]:
app=graph.compile()
app.invoke({
    "resume_path":"Software Engineer",
    "job_req":"AI Engineer",
    "session_id":"chat1"
})

### Task1 Done 
Partially done as the model now understands what the code is


The model to understand the resume updates is taking a long time hence you hav to do it one time, but for uodating every single time for the job is not right hece create a agent that has the 
1. Full text, which has the feature if FAISS, such that when the agent sees "okay this is the job title, i have to seacrh where the job title is in here", for that we use faiss.
2. Until the agent reads the whole document,the updates happen.

In [ ]:
model = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.9,
        api_key="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"
    )
small_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.6,
        api_key="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"
    )

@tool
def understand_project()->str:
    """Use this function to understand whether the original resume projects properly align with the Job Description"""
    prroject_text="""First one, Built an AI-powered EDA and visualization platform using Python, Streamlit, LangChain, and HuggingFace models. The system dynamically generated charts, handled missing data automatically, and enabled NLP-based dataset querying through custom AI agents. Demonstrates experience in LLM integration, AI workflow orchestration, intelligent automation, NLP systems, and interactive ML application development.
.Secondly, Developed an AI-driven migration assistant using LangChain, Streamlit, FAISS vector DB, and RAG pipelines to support legacy system modernization and framework/database migration tasks. Implemented semantic retrieval, prompt-engineered workflows, and contextual migration assistance, showcasing expertise in Generative AI, vector search systems, backend AI architecture, and intelligent developer tooling."""
    return prroject_text


@tool
def change_project(project_changes:str, reason:str)->None:
    """Use this tool to update the project section only if projects are totally different from the JD.Generate atleast 2 projects which strictly align with the JD"""
    latex_code =r"""\resumeSubHeadingListStart
    \resumeProjectHeading
        {\textbf{PROJECT NAME} $|$ }{}
        \resumeItemListStart
            \resumeItem{text explaining the project}
        \resumeItemListEnd 
    \resumeSubHeadingListEnd"""
    prompt=ChatPromptTemplate.from_messages([("assistant", "You are a intelligent resume maker, you would be provided with the content, of what should be placed in the latex template, you have to intelligently only provide the updated latex template"),
                                            ("user", """Here is the changes that have to be made:  {project_changes} along with the explanation of project in the resume item below in the latex code  due to reason {reason}, Here is the latex code  {latex_code}
                                            Strictly generate the output like the latex code, no extra explanation needed as we would be pasting whatever you give directly in the resume latex."""),
                                            ("ai", """%-----------PROJECTS-----------
\section""")])
        
    chain=prompt | small_llm | StrOutputParser()
    output=chain.invoke({
        "project_changes":project_changes,
        "latex_code":latex_code,
        "reason":reason
    })
    with open ("updates.txt","w") as f:
        f.write(output)
    return None



#@tool
#def understand_certificates()->str:

    return 
tools=[understand_project, change_project]
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an intelligent ATS and resume updation machine.
Based upon the requirements given, implement the changes in the original resume **only if needed**.
 First call understand project to understand the the project and help you tally 
"""
    ),
    ("human", "{JD}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])
agent = create_tool_calling_agent(
    llm=model,
    tools=tools,
    prompt=prompt
)
final_agent = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)
final_agent.invoke({

        "JD": """
    
Architect and lead the delivery of scalable, cloud-native software systems across backend and frontend layers. ﻿﻿Own end-to-end design and implementation of GenAl and LLM-powered platform capabilities in production. ﻿﻿Design and govern multi-agent Al architectures using MCP, A2A, and emerging agentic frameworks. ﻿﻿Define containerisation, orchestration, and deployment standards using Docker, Kubernetes, and Helm. ﻿﻿Lead architectural reviews, establish engineering best practices, and set technical direction for the team. ﻿﻿Drive cross-functional technical collaboration with product, data science, security, and infrastructure teams. Identify and resolve systemic technical risks across services, APIs, and data pipelines. ﻿﻿Mentor junior engineers through pairing, code reviews, and structured knowledge transfer. ﻿﻿Champion observability, reliability, and security practices across the platform. Familiarity with Agentic AI/Multi Agent orchestration frameworks like OpenAI agents SDK, LangGraph, etc is desired Experience with workflow management tools like Temporal, Airflow
    """
    })




        
    



> Entering new AgentExecutor chain...

Invoking: `understand_project` with `{}`


First one, Built an AI-powered EDA and visualization platform using Python, Streamlit, LangChain, and HuggingFace models. The system dynamically generated charts, handled missing data automatically, and enabled NLP-based dataset querying through custom AI agents. Demonstrates experience in LLM integration, AI workflow orchestration, intelligent automation, NLP systems, and interactive ML application development.
.Secondly, Developed an AI-driven migration assistant using LangChain, Streamlit, FAISS vector DB, and RAG pipelines to support legacy system modernization and framework/database migration tasks. Implemented semantic retrieval, prompt-engineered workflows, and contextual migration assistance, showcasing expertise in Generative AI, vector search systems, backend AI architecture, and intelligent developer tooling.
Invoking: `understand_project` with `{}`


First one, Built an AI-powered EDA and vi

{'JD': '\n    \nArchitect and lead the delivery of scalable, cloud-native software systems across backend and frontend layers. \ufeff\ufeffOwn end-to-end design and implementation of GenAl and LLM-powered platform capabilities in production. \ufeff\ufeffDesign and govern multi-agent Al architectures using MCP, A2A, and emerging agentic frameworks. \ufeff\ufeffDefine containerisation, orchestration, and deployment standards using Docker, Kubernetes, and Helm. \ufeff\ufeffLead architectural reviews, establish engineering best practices, and set technical direction for the team. \ufeff\ufeffDrive cross-functional technical collaboration with product, data science, security, and infrastructure teams. Identify and resolve systemic technical risks across services, APIs, and data pipelines. \ufeff\ufeffMentor junior engineers through pairing, code reviews, and structured knowledge transfer. \ufeff\ufeffChampion observability, reliability, and security practices across the platform. Familiarit

Instead of making the modek to perform Similary search which is performing really bad here, we coukd make the make the tool call the respective `str` which has the heading and data, for example when the model has to look onto the project, it would call the project tool which has the project details, when you return it the model would decide to call whether the change project has to be performed or not (due to the doc string in the function). Hence we solve the porblem of irrelevant text being loaded by the faiss, and lag it does initially when we load the notebook


So:

1. Make the project tool which has the project details in it, the model would get the correct code, hence it would be able to easily tally the correct code with the current job description.
 

In [ ]:
vc.similarity_search("PROJECTS")

[Document(id='c569db24-a085-41b4-8472-c1f81517dae2', metadata={'source': 'resume.txt'}, page_content='\\resumeSubheading\n{\\textbf{AIML Core Committee — Google Developer Group, Campus CVR}}{August 2024 – Present}\n{Core Committee Member}{}\n\\resumeItemListStart\n\\resumeItem{Conducted AI/ML bootcamps, interactive workshops, and events covering core algorithms and real-world applications; mentored students and collaborated with peers to build hands-on AI/ML project experience.}\n\\resumeItemListEnd\n\n\\resumeSubheading\n{\\textbf{AI Research Intern}}{May 2024 – July 2024}\n{AI Research Intern}{}\n\\resumeItemListStart\n\\resumeItem{Worked on core understanding of Transformer models, developing Agentic AI Transformers for enterprise needs, unlike generic LLMs; optimized architectures and deployment for a specific company’s AI use case.}\n\\resumeItemListEnd'),
 Document(id='dacfd554-7f89-457b-b9aa-cdff6d432d9e', metadata={'source': 'resume.txt'}, page_content='%-----------------------

In [1]:
from langchain_groq import ChatGroq
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.agents import AgentExecutor, create_tool_calling_agent

In [ ]:

model = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,
        api_key="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"
    )
small_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.6,
        api_key="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"
    )
projects="""
Built an AI-powered EDA and visualization platform using Python, Streamlit, LangChain, and HuggingFace models. The system dynamically generated charts, handled missing data automatically, and enabled NLP-based dataset querying through custom AI agents. Demonstrates experience in LLM integration, AI workflow orchestration, intelligent automation, NLP systems, and interactive ML application development.

Developed an AI-driven migration assistant using LangChain, Streamlit, FAISS vector DB, and RAG pipelines to support legacy system modernization and framework/database migration tasks. Implemented semantic retrieval, prompt-engineered workflows, and contextual migration assistance, showcasing expertise in Generative AI, vector search systems, backend AI architecture, and intelligent developer tooling.
"""

@tool
def understand_project(job_description:str)->str:
    """
    Analyze whether the existing projects align with the JD.
    AI, LangChain, RAG, FAISS, NLP, LLM and backend AI projects should be considered highly relevant for GenAI and AI engineering roles.
    Replace projects only if there is very low semantic similarity.
    """
    
    prompt=ChatPromptTemplate.from_messages([
        ("system",
        """
You are an ATS analyzer, who analyzes and understands how close are the projects of user related to the Job description

Return strictly in this format:

ALIGNMENT_SCORE: number

SHOULD_CHANGE_PROJECTS: YES or NO

REASON:
small explanation

        """),
        ("human","JD:\n{jd}\n\nProjects:\n{projects}")
    ])

    chain=prompt|small_llm|StrOutputParser()

    return chain.invoke({
        "jd":job_description,
        "projects":projects
    })

@tool
def change_project(project_changes:str,reason:str)->str:
    """
    Use this to change the projects based on the score given by understand project, if the score is less than 50 only then execute this
    """
    
    latex_code=r"""
\resumeSubHeadingListStart
\resumeProjectHeading
{\textbf{PROJECT NAME} $|$ }{}
\resumeItemListStart
\resumeItem{text explaining the project}
\resumeItemListEnd
\resumeSubHeadingListEnd
"""

    prompt=ChatPromptTemplate.from_messages([
        ("system",
        """
You are a resume latex generator.
Return only latex code.
Do not add explanations.
        """),
        ("human",
        """
Project Changes:
{project_changes}

Reason:
{reason}

Latex Template:
{latex_code}
        """)
    ])

    chain=prompt|small_llm|StrOutputParser()

    output=chain.invoke({
        "project_changes":project_changes,
        "reason":reason,
        "latex_code":latex_code
    })

    with open("updates.txt","w",encoding="utf-8") as f:
        f.write(output)

    return output

tools=[understand_project, change_project]

prompt=ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an intelligent ATS resume optimizer.

Your task is to optimize the resume according to the Job Description while preserving authenticity and avoiding unnecessary modifications.

Workflow:

1. Go to understand project. 
    IMPORTANT:
    - NEVER call change_project if SHOULD_CHANGE_PROJECTS is NO
    - NEVER call change_project if alignment_score >= 50
    - change_project may only be called once

Note: 
1. First analyze the current resume sections using the appropriate understanding tools. 
2. Measure semantic similarity between the resume content and the Job Description. 
3. Preserve sections that already have strong or moderate relevance. 
4. Only modify sections when there is very low domain or skill overlap. 
5. Prefer improving existing content over completely replacing it. 
6. Missing a few technologies or keywords alone is NOT enough reason to rewrite a section. 
7. Evaluate alignment based on: 
    - technical skills 
    - responsibilities 
    - tools and frameworks 
    - domain relevance 
    - transferable experience 
    - certifications 
    - engineering workflows 
8. Use semantic understanding instead of exact keyword matching. 
9. Only generate new content when the current content is fundamentally unrelated to the Job Description.



Your goal is intelligent resume optimization, not aggressive rewriting.
        """
    ),
    ("human","{JD}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent=create_tool_calling_agent(
    llm=model,
    tools=tools,
    prompt=prompt
)

final_agent=AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

jd="""
About the Role:
We’re hiring two front-end developers—one senior and one junior—to work on website builds, updates, and ongoing maintenance. Most of the work involves converting Figma designs into clean, responsive front-end code and supporting backend developers.

Senior Front-End Developer

Responsibilities:

Convert Figma designs into high-quality, responsive HTML/CSS
Handle complex layouts and UI components independently
Work closely with backend developers for integrations
Maintain and improve existing websites
Requirements:

Strong experience in HTML, CSS, JavaScript
Solid hands-on experience with Tailwind CSS (must-have)
Ability to deliver pixel-perfect, responsive layouts
Can work independently and manage tasks without supervision
Junior Front-End Developer

Responsibilities:

Convert Figma designs into responsive HTML/CSS with guidance
Make updates and edits to existing websites
Support senior developers and backend team
Requirements:

Good working knowledge of HTML and CSS (not basic)
Comfortable with Tailwind CSS
Good understanding and experience on JavaScript
Willingness to learn and improve
What We’re Looking For:

Attention to detail
Clean, structured code
Someone reliable who gets work done on time
Bonus (Good to Have):

Experience with Bootstrap or similar frameworks
Basic Git knowledge
Pay: ₹300,000.00 - ₹600,000.00 per year

Benefits:

Paid sick time
Application Question(s):
"""

analysis=final_agent.invoke({        "JD": """
  
Architect and lead the delivery of scalable, cloud-native software systems across backend and frontend layers. ﻿﻿Own end-to-end design and implementation of GenAl and LLM-powered platform capabilities in production. ﻿﻿Design and govern multi-agent Al architectures using MCP, A2A, and emerging agentic frameworks. ﻿﻿Define containerisation, orchestration, and deployment standards using Docker, Kubernetes, and Helm. ﻿﻿Lead architectural reviews, establish engineering best practices, and set technical direction for the team. ﻿﻿Drive cross-functional technical collaboration with product, data science, security, and infrastructure teams. Identify and resolve systemic technical risks across services, APIs, and data pipelines. ﻿﻿Mentor junior engineers through pairing, code reviews, and structured knowledge transfer. ﻿﻿Champion observability, reliability, and security practices across the platform. Familiarity with Agentic AI/Multi Agent orchestration frameworks like OpenAI agents SDK, LangGraph, etc is desired Experience with workflow management tools like Temporal, Airflow
    """
    },max_iterations=4, early_stopping_method="force")

# analysis=final_agent.invoke({
#     "JD":jd
# }, max_iterations=4, early_stopping_method="force")

print(analysis["output"])




> Entering new AgentExecutor chain...

Invoking: `understand_project` with `{'job_description': 'Architect and lead the delivery of scalable, cloud-native software systems across backend and frontend layers. \u200b\u200bOwn end-to-end design and implementation of GenAl and LLM-powered platform capabilities in production. \u200b\u200bDesign and govern multi-agent Al architectures using MCP, A2A, and emerging agentic frameworks. \u200b\u200bDefine containerisation, orchestration, and deployment standards using Docker, Kubernetes, and Helm. \u200b\u200bLead architectural reviews, establish engineering best practices, and set technical direction for the team. \u200b\u200bDrive cross-functional technical collaboration with product, data science, security, and infrastructure teams. Identify and resolve systemic technical risks across services, APIs, and data pipelines. \u200b\u200bMentor junior engineers through pairing, code reviews, and structured knowledge transfer. \u200b\u200bChampion